# 04: External Validation — Malicious Network Dataset

This notebook tests `external_validation_pipeline.py`: a standalone
validation pipeline that applies the **same methodology** as
`adaptrap_pipeline.py` (per-source-IP behavioral profiling,
silhouette-tuned Isolation Forest contamination, 70th-percentile
two-tier escalation) to `malicious_network_dataset`, an externally
labeled connection-level dataset.

**This is not a re-run of M3.** `malicious_network_dataset`'s schema
is incompatible with the CICHoneynet features M3 was trained on: no
per-packet timestamps (so `conn_rate`/`session_duration`/
`packets_per_second` can't be reconstructed), no TCP-flag `Info`
string (so `syn/ack/fin/rst/psh_ratio` can't be reconstructed), and
`length` is on a different scale entirely (this dataset's max length
is ~100,000 vs. CICHoneynet's ~1,514, the standard Ethernet MTU).
Forcing this data through M3's fitted scaler would produce
meaningless scores. Instead, this notebook independently applies the
same unsupervised methodology using only the features this dataset's
schema actually supports, and checks the result against ground truth
only at the final step.


In [1]:
import sys
sys.path.insert(0, '.')
from external_validation_pipeline import (
    load_unlabeled, build_ip_profiles, three_way_split,
    tune_contamination, generate_rules, evaluate_against_ground_truth,
    FEATURE_COLS, TARGET_LOCAL_IP,
)
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)
print("Feature columns used:", FEATURE_COLS)
print("Target host filtered to:", TARGET_LOCAL_IP)


Feature columns used: ['total_packets', 'unique_local_ports', 'unique_remote_ports', 'avg_length', 'std_length', 'max_length', 'payload_ratio', 'payload_diversity']
Target host filtered to: 165.227.180.71


## 1. Load the unlabeled dataset

Loads `malicious_network_dataset_nolabel.csv` -- the model never
sees the `class` column at any point in this section. Cleaning keeps
only real TCP connections to the actual target host, matching the
inbound-traffic-only filter `adaptrap_pipeline.load_and_clean` applies
to CICHoneynet captures.


In [2]:
df = load_unlabeled('malicious_network_dataset_nolabel.csv')
print("Cleaned connection rows:", len(df))
print("Unique attacker IPs:", df['remote_ip'].nunique())
df.head()


Cleaned connection rows: 24987
Unique attacker IPs: 3032


,protocol,remote_ip,remote_port,local_ip,local_port,md5_hash,sha512_hash,length,data_hex
0,tcp,35.203.211.180,65428.0,165.227.180.71,45876.0,1fb4aeaab94ca27d2e5dfaa47e11a6fb,a6d4f36a2a8d5b5ea7e6afe91d4a80e7a9ae2129ebf479...,207.0,16030100ca010000c60303918984ed51d5c0b8d4cfad73...
1,tcp,180.93.172.180,57902.0,165.227.180.71,5901.0,d41d8cd98f00b204e9800998ecf8427e,cf83e1357eefb8bdf1542850d66d8007d620e4050b5715...,0.0,NaN
2,tcp,81.17.19.66,47564.0,165.227.180.71,9999.0,19b893b938ace1defe7d090e510f0618,f060846cbf02e31706d5d0fe781d70071bf76c903ef947...,3.0,50100.0
3,tcp,94.23.145.155,36536.0,165.227.180.71,62934.0,59b490c4ab003464ca03428b3fc63222,922450f93e933de877934cee97339ae6a22b39c12cbb31...,199.0,474554202f20485454502f312e310d0a486f73743a2031...
4,tcp,94.156.177.148,65462.0,165.227.180.71,3846.0,c1896a67f5b92c47df9185f85bcaaf22,9282e1f706f743b863c295c629f8b2b05b9e6aa152f2af...,47.0,0300002f2ae00000000000436f6f6b69653a206d737473...


## 2. Build per-source-IP behavioral profiles

One row per `remote_ip`, aggregated the same way
`adaptrap_pipeline.build_ip_profiles` aggregates per `source_ip` --
just restricted to the feature subset this schema supports.


In [3]:
profiles = build_ip_profiles(df)
print("Profile table shape:", profiles.shape)
profiles[FEATURE_COLS].describe()


Profile table shape: (3032, 9)


,total_packets,unique_local_ports,unique_remote_ports,avg_length,std_length,max_length,payload_ratio,payload_diversity
count,3032.000000,3032.000000,3032.000000,3032.000000,3032.000000,3032.000000,3032.000000,3032.000000
mean,8.241095,2.987797,6.221966,183.104905,56.393133,239.160950,0.746055,0.734476
std,146.582848,38.894338,103.059335,2183.885472,1045.984570,2570.286136,0.402373,0.395383
min,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,18.156250,0.000000,25.750000,0.500000,0.500000
50%,2.000000,1.000000,2.000000,115.000000,0.000000,207.000000,1.000000,1.000000
75%,4.000000,2.000000,3.000000,225.000000,67.663222,243.000000,1.000000,1.000000
max,7090.000000,1691.000000,4970.000000,100000.000000,57531.799624,100000.000000,1.000000,1.000000


## 3. Train / validation / holdout split

Plain random 70/15/15 split (unlike
`adaptrap_pipeline.stratified_three_way_split`, this can't stratify on
dominant protocol -- protocol is a constant `'tcp'` here after
cleaning, so there's nothing to stratify on).


In [4]:
train_idx, val_idx, test_idx = three_way_split(profiles)
print(f"train={len(train_idx)}  val={len(val_idx)}  holdout={len(test_idx)}")


train=2122  val=455  holdout=455


## 4. Tune contamination (silhouette-selected, validation set only)

Identical selection logic to `adaptrap_pipeline.tune_contamination`:
sweep candidate contamination values, score each on the validation set
by Flagging Rate and Silhouette Coefficient, select by silhouette.
Ground truth is not touched -- this is purely unsupervised, exactly
mirroring the original methodology.


In [5]:
contamination, model, scaler, sweep = tune_contamination(
    profiles, FEATURE_COLS, train_idx, val_idx
)
print(sweep)
print("\nSelected contamination:", contamination)


    contamination  flagging_rate  silhouette
0          0.0100       0.010989    0.502402
1          0.0125       0.010989    0.502402
2          0.0150       0.015385    0.451512
3          0.0175       0.017582    0.440566
4          0.0200       0.017582    0.440566
5          0.0500       0.046154    0.337951
6          0.0750       0.061538    0.333798
7          0.1000       0.087912    0.317712
8          0.1250       0.123077    0.280119
9          0.1500       0.142857    0.259403
10         0.2000       0.197802    0.267096

Selected contamination: 0.01


## 5. Score the holdout set and generate ALLOW / ESCALATE_TO_ANALYST rules

Same two-tier logic as `adaptrap_pipeline.generate_rules`: escalate
anything at or above the 70th percentile of the holdout's own anomaly
score distribution. No auto-block tier here either.


In [6]:
rules_df = generate_rules(model, scaler, profiles, FEATURE_COLS, test_idx,
                           label="External Validation Holdout")
rules_df.head(10)



--- External Validation Holdout ---
ESCALATE_TO_ANALYST: 143, ALLOW: 312


,remote_ip,anomaly_score,action
0,94.23.145.155,0.096878,ESCALATE_TO_ANALYST
1,45.84.89.3,0.083873,ESCALATE_TO_ANALYST
2,79.137.198.113,0.074370,ESCALATE_TO_ANALYST
3,13.58.97.162,0.072658,ESCALATE_TO_ANALYST
4,180.93.172.180,0.044129,ESCALATE_TO_ANALYST
5,80.66.83.47,0.042012,ESCALATE_TO_ANALYST
6,45.79.155.77,0.033498,ESCALATE_TO_ANALYST
7,80.66.83.114,0.022513,ESCALATE_TO_ANALYST
8,179.43.133.162,0.015335,ESCALATE_TO_ANALYST
9,64.227.45.231,0.009283,ESCALATE_TO_ANALYST


## 6. Export Isolation Forest results to CSV

Same pattern as `generated_firewall_rules_M3__Adaptive__on_Batch_3_holdout.csv`
in the main pipeline: the model's ALLOW / ESCALATE_TO_ANALYST decisions
are written out as a standalone CSV before anything is compared against
ground truth.


In [7]:
rules_df.to_csv('external_validation_rules_holdout.csv', index=False)
print("Exported:", len(rules_df), "rows -> external_validation_rules_holdout.csv")


Exported: 455 rows -> external_validation_rules_holdout.csv


## 7. Reload the exported CSV and compare against the labeled dataset

This reads the Isolation Forest output back from disk (not from the
in-memory `rules_df`) and joins it against `malicious_network_dataset.csv`
-- the labeled ground truth. This is the only step in this notebook
that reads the `class` column. An IP counts as ground-truth
`malicious` if *any* of its connections were labeled malicious -- a
deliberate, security-conservative aggregation choice, since the label
is per-connection but the model scores per-IP.


In [8]:
exported_rules_df = pd.read_csv('external_validation_rules_holdout.csv')
merged, metrics = evaluate_against_ground_truth(exported_rules_df, 'malicious_network_dataset.csv')

print(f"Holdout n = {metrics['n']}")
print(f"Majority-class baseline accuracy: {metrics['baseline_majority_class_accuracy']:.3f}")
print(f"\nConfusion matrix {metrics['confusion_matrix_labels']}:")
print(metrics['confusion_matrix'])
print(f"\nPrecision (malicious): {metrics['precision_malicious']:.3f}")
print(f"Recall (malicious):    {metrics['recall_malicious']:.3f}")
print(f"F1 (malicious):        {metrics['f1_malicious']:.3f}")
print(f"Accuracy:              {metrics['accuracy']:.3f}")


Holdout n = 455
Majority-class baseline accuracy: 0.655

Confusion matrix ['malicious', 'benign']:
[[ 86 212]
 [ 57 100]]

Precision (malicious): 0.601
Recall (malicious):    0.289
F1 (malicious):        0.390
Accuracy:              0.409


## 8. Summary

The escalation decisions from this schema-restricted validation model
should be compared against the **majority-class baseline accuracy**
above, not against 100%. If accuracy sits at or below that baseline,
it indicates the available feature set -- not the underlying
methodology -- is the limiting factor: this dataset lacks the
timestamp-derived (`conn_rate`, `session_duration`,
`packets_per_second`) and TCP-flag-derived
(`syn/ack/fin/rst/psh_ratio`) features that carry much of the
discriminative signal in the main CICHoneynet-based pipeline.

This result should be reported as a supplementary, schema-limited
external validation, not as a direct measure of M3's real-world
performance.
